## User-Based + Item-Based

In [22]:
import ast
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from rapidfuzz import process, fuzz

### Load Steam Dataset

In [23]:
# Подгрузим Steam Dataset, возьмем только записи тех кто уже поиграл в игру.

steam_raw = pd.read_csv('../SteamDataset200K/steam-200k.csv', header=None, names=['user_id', 'game', 'behavior', 'hours', 'zero'])

df = steam_raw[steam_raw['behavior'] == 'play'].copy()
df = df[['user_id', 'game', 'hours']].reset_index(drop=True)

df.head()

,user_id,game,hours
0,151603712,The Elder Scrolls V Skyrim,273.0
1,151603712,Fallout 4,87.0
2,151603712,Spore,14.9
3,151603712,Fallout New Vegas,12.1
4,151603712,Left 4 Dead 2,8.9


In [24]:
# Конвертируем игровые часы в рейтинг, чтобы метрика стала более читаемой.
# Масштабируем логарифмически.

def hours_to_rating(hours_series):
    log_h = np.log1p(hours_series)
    scaler = MinMaxScaler(feature_range=(1, 10))
    ratings = scaler.fit_transform(log_h.values.reshape(-1, 1)).flatten()

    return np.round(ratings, 2)


df['rating'] = hours_to_rating(df['hours'])

df.head(10)

,user_id,game,hours,rating
0,151603712,The Elder Scrolls V Skyrim,273.0,6.35
1,151603712,Fallout 4,87.0,5.25
2,151603712,Spore,14.9,3.59
3,151603712,Fallout New Vegas,12.1,3.40
4,151603712,Left 4 Dead 2,8.9,3.13
5,151603712,HuniePop,8.5,3.09
6,151603712,Path of Exile,8.1,3.05
7,151603712,Poly Bridge,7.5,2.98
8,151603712,Left 4 Dead,3.3,2.32
9,151603712,Team Fortress 2,2.8,2.20


In [25]:
# Делаем датасет менее разряженным (это может повлечь проблемы: непопулярные игры не будут рекомендоваться, но мы подумаем над этим позже)

MIN_GAMES_PER_USER = 3
MIN_USERS_PER_GAME = 5

user_counts = df.groupby('user_id')['game'].count()
game_counts = df.groupby('game')['user_id'].count()

active_users = user_counts[user_counts >= MIN_GAMES_PER_USER].index
popular_games = game_counts[game_counts >= MIN_USERS_PER_GAME].index

df_filtered = df[df['user_id'].isin(active_users) & df['game'].isin(popular_games)]

user_item_matrix = df_filtered.pivot_table(
    index='user_id',
    columns='game',
    values='rating',
    aggfunc='mean'
).fillna(0)

user_item_matrix

game,1... 2... 3... KICK IT! (Drop That Beat Like an Ugly Baby),100% Orange Juice,12 Labours of Hercules,12 Labours of Hercules II The Cretan Bull,140,3DMark,404Sight,60 Seconds!,7 Days to Die,8BitBoy,...,Zeno Clash,Zombie Army Trilogy,Zombie Driver,Zombie Panic Source,Zombies Monsters Robots,ibb & obb,resident evil 4 / biohazard 4,sZone-Online,the static speaks my name,theHunter
user_id,,,,,,,,,,,,,,,,,,,,,
5250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
76767,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
86540,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
229911,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
298950,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.55,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306547522,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
306971738,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
308695132,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### User-Based

In [26]:
def user_based_rec(user_id, user_item_matrix, n_neighbors=20, top_n=20):
    if user_id not in user_item_matrix.index:
        return None

    sim_matrix = cosine_similarity(user_item_matrix.values)
    sim_df = pd.DataFrame(sim_matrix, index=user_item_matrix.index, columns=user_item_matrix.index)

    user_sim = sim_df[user_id].drop(user_id).sort_values(ascending=False)
    neighbors = user_sim.head(n_neighbors)

    user_games = user_item_matrix.loc[user_id]
    unplayed = user_games[user_games == 0].index

    neighbor_ratings = user_item_matrix.loc[neighbors.index, unplayed]
    weights = neighbors.values.reshape(-1, 1)

    weighted_sum = (neighbor_ratings * weights).sum(axis=0)
    weight_total = (neighbor_ratings > 0).astype(int).T.dot(weights.flatten())
    weight_total = weight_total.replace(0, np.nan)

    predicted = (weighted_sum / weight_total).dropna().sort_values(ascending=False)

    result = predicted.head(top_n).reset_index()
    result.columns = ['game', 'ub_score']

    return result


sample_user = user_item_matrix.index[0]
games_sample_user = user_item_matrix.loc[sample_user]
played = games_sample_user[games_sample_user != 0]
ub_recs = user_based_rec(sample_user, user_item_matrix)

played, ub_recs

(game
 Alien Swarm                 2.63
 Cities Skylines             5.74
 Deus Ex Human Revolution    4.93
 Dota 2                      1.08
 Portal 2                    3.51
 Team Fortress 2             1.48
 Name: 5250, dtype: float64,
                                            game  ub_score
 0                         Counter-Strike Source  6.550000
 1                          Farming Simulator 15  5.250000
 2                          Realm of the Mad God  5.110000
 3                                Clicker Heroes  5.020000
 4                            Half-Life 2 Update  4.970000
 5                                   Garry's Mod  4.960000
 6                            Tabletop Simulator  4.860000
 7               Never Alone (Kisima Ingitchuna)  4.810000
 8                                    Watch_Dogs  4.780000
 9   Grand Theft Auto Episodes from Liberty City  4.530000
 10                                  Train Fever  4.490000
 11      Call of Duty Black Ops II - Multiplayer  4.4

### Load PlayMyData

In [27]:
pc = pd.read_csv('../PlayMyData/all_games_PC.csv')
ps = pd.read_csv('../PlayMyData/all_games_PlayStation.csv')
genres_df = pd.read_csv('../PlayMyData/genres.csv')

games_df = pd.concat([pc, ps], ignore_index=True).drop_duplicates(subset='id').reset_index(drop=True)

games_df['rating'] = pd.to_numeric(games_df['rating'], errors='coerce')

games_df.head()

,genres,id,name,platforms,summary,storyline,rating,main,extra,completionist,review_score,review_count,people_polled
0,[5],274203,Short 'n Quick,[6],this is a techstyled map taking place in a war...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,[5],256819,Caverns of Darkness,[6],The final rift was closed and the Hell War was...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"[26, 31]",232860,"Nu, pogodi! Vypusk 1: Pogonya",[6],The wolf decides to take revenge on the Hare f...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"[12, 15, 16]",228979,Power Dolls 5,[6],The fifth entry in Kogado Studios allfemale me...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[2],225648,Kapsyljakt med Anki & Pytte,[6],A pointandclick game based on the Swedish chil...,Missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
def parse_genres(value):
    if not isinstance(value, str):
        return []
    value = value.strip()
    if value == '' or value == 'Missing':
        return []
    try:
        result = ast.literal_eval(value)

        return result if isinstance(result, list) else []
    except Exception:
        return []


games_df['genres_list'] = games_df['genres'].apply(parse_genres)

#One-hot encoding
all_genre_ids = genres_df['genre_id'].tolist()
for gid in all_genre_ids:
    games_df[f'genre_{gid}'] = games_df['genres_list'].apply(lambda x: int(gid in x))

num_features = ['rating', 'review_score', 'main', 'extra', 'completionist']
for column in num_features:
    games_df[column] = pd.to_numeric(games_df[column], errors='coerce')
    games_df[column] = games_df[column].fillna(games_df[column].median())
    max_value = games_df[column].max()

    if max_value > 0:
        games_df[column] = games_df[column] / max_value

games_df.head(10)

,genres,id,name,platforms,summary,storyline,rating,main,extra,completionist,...,genre_26,genre_25,genre_30,genre_31,genre_33,genre_34,genre_32,genre_35,genre_36,genre_2
0,[5],274203,Short 'n Quick,[6],this is a techstyled map taking place in a war...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,[5],256819,Caverns of Darkness,[6],The final rift was closed and the Hell War was...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,"[26, 31]",232860,"Nu, pogodi! Vypusk 1: Pogonya",[6],The wolf decides to take revenge on the Hare f...,Missing,0.7,0.000309,0.0,0.0,...,1,0,0,1,0,0,0,0,0,0
3,"[12, 15, 16]",228979,Power Dolls 5,[6],The fifth entry in Kogado Studios allfemale me...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,[2],225648,Kapsyljakt med Anki & Pytte,[6],A pointandclick game based on the Swedish chil...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,1
5,[10],204650,Rage Rally,[6],Missing,Missing,0.7,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
6,[34],194713,Ripple: Blue Seal he Youkoso,[6],Main character loses a job and applies for an ...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,1,0,0,0,0
7,[31],73345,The Mystery of the Nautilus,[6],This adventure game was inspired by the Jules ...,The story begins aboard the USS Shark a resear...,0.7,0.000309,0.0,0.0,...,0,0,0,1,0,0,0,0,0,0
8,"[2, 31]",71917,JumpStart: Animal Adventures,"[6, 14]",Educational game for teaching kids about wild ...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,1,0,0,0,0,0,1
9,Missing,61935,Farland Symphony,[6],A spinoff title in the Farland Story RPG franc...,Missing,0.7,0.000309,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
# Решаем проблему исключения нишевых игр
# Для Item-Based расширяем матрицу сходства, игры из Steam датасета используем как сиды (игры юзера), а при нахождении похожих игр, берем все из PlayMyData (85k).
# Это позволит рекомендовать нишевые игры, даже если их вообще не было в датасете Steam.

steam_games_names = set(user_item_matrix.columns)

genre_columns = [f'genre_{gid}' for gid in all_genre_ids]
num_columns = ['rating', 'review_score', 'main', 'extra', 'completionist']
feature_columns = genre_columns + num_columns

GENRE_WEIGHT = 3.0
NUMERIC_WEIGHT = 1.5

all_content_games = games_df.copy().drop_duplicates(subset='name').set_index('name')
feature_matrix_all = all_content_games[feature_columns].fillna(0).values.copy().astype(float)

n_genre = len(genre_columns)
feature_matrix_all[:, :n_genre] *= GENRE_WEIGHT
feature_matrix_all[:, :n_genre] *= NUMERIC_WEIGHT

steam_in_playmydata = [g for g in steam_games_names if g in all_content_games.index]
steam_index = [all_content_games.index.get_loc(g) for g in steam_in_playmydata]
steam_vectors = feature_matrix_all[steam_index]

item_sim_matrix = cosine_similarity(feature_matrix_all, steam_vectors)
item_sim_df = pd.DataFrame(
    item_sim_matrix,
    index=all_content_games.index,
    columns=steam_in_playmydata
)

item_sim_df

,Euro Truck Simulator,Gunpoint,Legend of Dungeon,Sid Meier's Railroads!,Football Superstars,The Tiny Bang Story,Godus,Project CARS,Shadowgrounds Survivor,Wings of Prey,...,The Evil Within,Football Manager 2010,Echo of Soul,Five Nights at Freddy's 2,The Misadventures of P.B. Winterbottom,Uncrowded,Men of War,LEGO The Lord of the Rings,Boson X,Dustforce
name,,,,,,,,,,,,,,,,,,,,,
Short 'n Quick,0.021123,0.021689,0.020794,0.030166,0.023455,0.019069,0.014696,0.025612,0.714827,0.036464,...,0.587170,0.027882,0.032107,0.019974,0.018616,0.026082,0.027364,0.039154,0.022334,0.043018
Caverns of Darkness,0.021123,0.021689,0.020794,0.030166,0.023455,0.019069,0.014696,0.025612,0.714827,0.036464,...,0.587170,0.027882,0.032107,0.019974,0.018616,0.026082,0.027364,0.039154,0.022334,0.043018
"Nu, pogodi! Vypusk 1: Pogonya",0.015079,0.326683,0.416826,0.021534,0.016744,0.325265,0.010491,0.018283,0.020635,0.026030,...,0.419157,0.019904,0.022920,0.325768,0.325011,0.018619,0.019534,0.713824,0.015943,0.030709
Power Dolls 5,0.012351,0.267593,0.341431,0.418316,0.013715,0.011150,0.242099,0.014976,0.016902,0.021322,...,0.015181,0.343957,0.584465,0.266844,0.010885,0.581919,0.343832,0.022895,0.013059,0.025154
Kapsyljakt med Anki & Pytte,0.021123,0.021689,0.020794,0.030166,0.023455,0.455642,0.414030,0.025612,0.028906,0.036464,...,0.025962,0.027882,0.032107,0.456347,0.018616,0.026082,0.027364,0.039154,0.022334,0.043018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Magical Drop 3,0.027777,0.458313,0.026759,0.039080,0.027433,0.455499,0.019090,0.033456,0.037432,0.047903,...,0.587572,0.036397,0.041742,0.025969,0.455028,0.031344,0.035793,0.050371,0.029191,0.056524
Tenchu 2: Birth of the Stealth Assassins,0.026819,0.458339,0.582634,0.037849,0.027174,0.455643,0.018478,0.032348,0.036255,0.046258,...,0.587662,0.035213,0.040396,0.456523,0.455186,0.030858,0.034597,0.998681,0.028218,0.054583
Dragon Warrior VII,0.016268,0.327385,0.818653,0.023428,0.019198,0.325816,0.011396,0.019799,0.022454,0.028098,...,0.419904,0.021563,0.714188,0.326402,0.325527,0.711560,0.021138,0.715102,0.017259,0.033145


In [30]:
FUZZY_THRESHOLD = 93

BLACKLIST = {'Besiege', 'Jamestown', 'Color Symphony', 'Goat Simulator', 'Borderlands 2 RU', 'Ragnarok', 'Knights of Pen and Paper +1'}

playmydata_names = all_content_games.index.tolist()
unmatched = steam_games_names - set(all_content_games.index)

fuzzy_map = {}
for steam_name in unmatched:
    if steam_name in BLACKLIST:
        continue

    match, score, _ = process.extractOne(steam_name,
                                         playmydata_names,
                                         scorer=fuzz.token_sort_ratio)
    if score >= FUZZY_THRESHOLD:
        fuzzy_map[steam_name] = match

fuzzy_rows = []
for steam_name, playmydata_name in fuzzy_map.items():
    if playmydata_name in all_content_games.index:
        row = all_content_games.loc[playmydata_name].copy()
        row.name = steam_name
        fuzzy_rows.append(row)

if fuzzy_rows:
    fuzzy_df = pd.DataFrame(fuzzy_rows)
    all_content_games = pd.concat([all_content_games, fuzzy_df])

len(all_content_games)

84030

### Item-Based

In [31]:
def item_based_rec(user_id, user_item_matrix, item_sim_df, n_neighbors=20, top_n=20, sim_threshold=0.7, min_seeds=2):
    if user_id not in user_item_matrix.index:
        return None

    user_games = user_item_matrix.loc[user_id]
    played = user_games[user_games != 0]
    played_known = played[played.index.isin(item_sim_df.columns)]

    if len(played_known) == 0:
        return None

    already_played = set(played.index)
    candidates = [g for g in item_sim_df.index if g not in already_played]

    sim_block = item_sim_df.loc[candidates, played_known.index].values.copy()

    for i in range(len(sim_block)):
        row = sim_block[i]
        threshold_idx = np.argsort(row)[:-n_neighbors]
        row[threshold_idx] = 0

    seeds_above_threshold = (sim_block >= sim_threshold).sum(axis=1)
    mask = seeds_above_threshold >= min_seeds

    sim_block = sim_block[mask]
    filtered_candidates = [c for c, m in zip(candidates, mask) if m]

    ratings = played_known.values
    weighted_sum = sim_block.dot(ratings)
    weight_total = sim_block.sum(axis=1)
    weight_total[weight_total == 0] = np.nan

    result = pd.Series(weighted_sum, index=filtered_candidates).dropna()
    result = result.sort_values(ascending=False).head(top_n).reset_index()
    result.columns = ['game', 'ib_score']

    return result


sample_user = user_item_matrix.index[0]
games_sample_user = user_item_matrix.loc[sample_user]
played = games_sample_user[games_sample_user != 0]

ib_recs = item_based_rec(sample_user, user_item_matrix, item_sim_df)

played, ib_recs

(game
 Alien Swarm                 2.63
 Cities Skylines             5.74
 Deus Ex Human Revolution    4.93
 Dota 2                      1.08
 Portal 2                    3.51
 Team Fortress 2             1.48
 Name: 5250, dtype: float64,
                                                game  ib_score
 0                 Halo: The Master Chief Collection  5.958774
 1                                   Resident Evil 4  5.958244
 2                                       Half-Life 2  5.958231
 3                                    Counter-Strike  5.958055
 4                                      Doom Eternal  5.957872
 5                                 The Ultimate Doom  5.957798
 6                                 Unreal Tournament  5.957736
 7                                          Returnal  5.957657
 8   No One Lives Forever 2: A Spy in H.A.R.M.'s Way  5.957594
 9                                            Halo 3  5.957589
 10         Medal of Honor: Allied Assault War Chest  5.957574
 11  

### Final Model (User-Based + Item-Based)

In [32]:
# alpha - вес User-Based оценки
# beta - вес Item-Based оценки
# boost_factor - множитель для игр из пересечения User-Based и Item-Based
# discount_factor - множитель для игр только из одного

def hybrid_rec(user_id, user_item_matrix, item_sim_df, n_neighbours=20, top_n=20,
               alpha=0.8, beta=0.2, boost_factor=1.3, discount_factor=0.7):
    ub = user_based_rec(user_id, user_item_matrix, n_neighbours, top_n)
    ib = item_based_rec(user_id, user_item_matrix, item_sim_df, n_neighbours, top_n)

    if ub is None or ib is None:
        return None

    def normalize(series):
        mn, mx = series.min(), series.max()

        return (series - mn) / (mx - mn + 1e-9)

    ub['ub_score'] = normalize(ub['ub_score'])
    ib['ib_score'] = normalize(ib['ib_score'])

    merged = pd.merge(ub, ib, on='game', how='outer')

    both = merged.dropna(subset=['ub_score', 'ib_score']).copy()
    both['final_score'] = (alpha * both['ub_score'] + beta * both['ib_score']) * boost_factor

    only_ub = merged[merged['ib_score'].isna()].copy()
    only_ub['final_score'] = only_ub['ub_score'] * discount_factor

    only_ib = merged[merged['ub_score'].isna()].copy()
    only_ib['final_score'] = only_ib['ib_score'] * discount_factor

    result = pd.concat([both, only_ub, only_ib], ignore_index=True)
    result = result[['game', 'final_score']].sort_values('final_score', ascending=False).head(top_n).reset_index(drop=True)

    result.index += 1

    return result


sample_user = user_item_matrix.index[0]
games_sample_user = user_item_matrix.loc[sample_user]
played = games_sample_user[games_sample_user != 0]

hybrid_recs = hybrid_rec(sample_user, user_item_matrix, item_sim_df)

played, hybrid_recs

(game
 Alien Swarm                 2.63
 Cities Skylines             5.74
 Deus Ex Human Revolution    4.93
 Dota 2                      1.08
 Portal 2                    3.51
 Team Fortress 2             1.48
 Name: 5250, dtype: float64,
                                                game  final_score
 1                             Counter-Strike Source     0.700000
 2                 Halo: The Master Chief Collection     0.700000
 3                                   Resident Evil 4     0.478233
 4                                       Half-Life 2     0.472737
 5                                    Counter-Strike     0.399179
 6                              Farming Simulator 15     0.373835
 7                              Realm of the Mad God     0.338710
 8                                      Doom Eternal     0.322538
 9                                    Clicker Heroes     0.316129
 10                               Half-Life 2 Update     0.303584
 11                                

### Оффлайн оценивание

In [38]:
def train_test_split_matrix(user_item_matrix, test_size=0.2, random_state=42):
    np.random.seed(random_state)
    train_matrix = user_item_matrix.copy()
    test_ground_truth = {}

    for user_id in user_item_matrix.index:
        interacted_items = user_item_matrix.columns[user_item_matrix.loc[user_id] > 0].tolist()

        if len(interacted_items) >= 2:
            n_test = max(1, int(len(interacted_items) * test_size))
            test_items = list(np.random.choice(interacted_items, size=n_test, replace=False))

            test_ground_truth[user_id] = test_items
            train_matrix.loc[user_id, test_items] = 0.0

    return train_matrix, test_ground_truth

train_matrix, test_ground_truth = train_test_split_matrix(user_item_matrix, test_size=0.2)
print(f'Пользователей для валидации: {len(test_ground_truth)}')

Пользователей для валидации: 3458


In [39]:
# Пересчитываем косинусное сходство между играми на основе train данных

train_item_sim_matrix = cosine_similarity(train_matrix.T)
train_item_sim_df = pd.DataFrame(item_sim_matrix, index=train_matrix.columns, columns=train_matrix.columns)

Метрики, которые считаем:
* `Precision`: из всех объектов, которые алгоритм выдал в топ-k, какая доля оказалась реально интересна юзеру?
* `Recall`: какую долю из всех своих реальных (из тестовой выборки) интересов юзер смог найти в предложенном топ-k списке?
* `Hit Rate`: какая вероятность, что юзер, открыв список из топ-k объектов, найдет там хотя бы одну интересную игру?
* `NDCG`: насколько хорошо отранжирован наш топ-k, находятся ли угаданные игры в самом верху выдачи или в самом низу?
* `MRR`: на какой в среднем позиции юзер встречает самую первую подходящую ему рекомендацию?

In [40]:
def evaluate(train_mat, test_gt, rec_function, similarity_df=None, k=10):
    metrics = {
        'precision': [], 'recall': [], 'hit_rate': [], 'ndcg': [], 'mrr': []
    }

    for user_id, ground_truth in test_gt.items():
        try:
            if similarity_df is not None:
                rec_result = rec_function(user_id, train_mat, similarity_df)
            else:
                rec_result = rec_function(user_id, train_mat)

            if rec_result is None or (isinstance(rec_result, pd.DataFrame) and rec_result.empty):
                continue

            if isinstance(rec_result, pd.DataFrame):
                if 'game' in rec_result.columns:
                    recommended_items = rec_result['game'].tolist()
                else:
                    recommended_items = rec_result.iloc[:, 0].tolist()
            else:
                recommended_items = list(rec_result)

        except Exception:
            try:
                rec_result = rec_function(user_id, train_mat, train_item_sim_df)
                recommended_items = rec_result['game'].tolist() if isinstance(rec_result, pd.DataFrame) else list(rec_result)
            except Exception:
                continue

        recommended_items = recommended_items[:k]
        if not recommended_items:
            continue

        hits = [1 if item in ground_truth else 0 for item in recommended_items]
        num_hits = sum(hits)

        precision = num_hits / k
        recall = num_hits / len(ground_truth)
        hit_rate = 1 if num_hits > 0 else 0

        dcg = sum([hits[i] / np.log2(i + 2) for i in range(len(hits))])
        idcg = sum([1 / np.log2(i + 2) for i in range(min(len(ground_truth), k))])
        ndcg = dcg / idcg if idcg > 0 else 0

        mrr = 0
        for i, hit in enumerate(hits):
            if hit == 1:
                mrr = 1 / (i + 1)
                break

        metrics['precision'].append(precision)
        metrics['recall'].append(recall)
        metrics['hit_rate'].append(hit_rate)
        metrics['ndcg'].append(ndcg)
        metrics['mrr'].append(mrr)

    if not metrics['precision']:
        return {"error": "Модель не вернула рекомендаций"}

    return {k_metric: round(np.mean(v), 4) for k_metric, v in metrics.items()}

np.random.seed(42)
result_ub = evaluate(train_matrix, test_ground_truth, user_based_rec, similarity_df=None, k=10)
result_ib = evaluate(train_matrix, test_ground_truth, item_based_rec, similarity_df=train_item_sim_df, k=10)
result_hybrid = evaluate(train_matrix, test_ground_truth, hybrid_rec, similarity_df=train_item_sim_df, k=10)

metrics_df = pd.DataFrame({
    'User-Based': result_ub,
    'Item-Based': result_ib,
    'Hybrid': result_hybrid
}).T

metrics_df

,precision,recall,hit_rate,ndcg,mrr
User-Based,0.0320,0.1809,0.2834,0.0980,0.0946
Item-Based,0.0333,0.0111,0.3333,0.0734,0.3333
Hybrid,0.0320,0.1806,0.2831,0.0979,0.0945


* Алгоритм Item-Based показал самый высокий `Hit Rate` (0.333) и такой же `MRR` (0.333). Это говорит о том, что когда Item-Based угадывает скрытую игру, он ставит ее строго на 1-е место в списке рекомендаций. Однако `Recall` у алгоритма слишком низкий (0.0111). Модель предлагает очень похожие игры на те, что пользователь уже запускал и способна точечно угадать по крайней мере 1 позицию, однако она не способна покрыть все разнообразие интересов пользователя.
* Алгоритм User-Based продемонстрировал значительно более высокий `Recall` (0.181) и `NDCG` (0.098). Он успешно находит скрытые игры, опираясь на схожие долгосрочные паттерны других людей. Но метрика `MRR` (0.0946) показывает, что релевантные игры размазаны по списку топ-k игр и чаще находятся ближе к концу выдачи.
* В Hybrid модели практически дублируются метрики User-Based подхода. Вероятнее всего, из-за разных масштабов score'ов (`ub_score`, `ib_score`) или весовых коэффициентов, User-Based подход подавляет Item-Based.